## Downloading Models

In [ ]:
import os
os.environ["HF_HOME"] = "/mnt/lustre/koa/scratch/kantas/hf_home"

from transformers import AutoModelForCausalLM, AutoTokenizer

# Specify the model name
tiny_model_name = "Qwen/Qwen2.5-0.5B"
medium_model_name = "Qwen/Qwen2.5-3B-Instruct"

# Download the model and tokenizer
for model_name in (tiny_model_name, medium_model_name):
    model = AutoModelForCausalLM.from_pretrained(model_name, trust_remote_code=True)
    tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)

    target_directory = "../models/"+model_name.replace("/", "_")
    tokenizer.save_pretrained(target_directory)
    model.save_pretrained(target_directory)

## Qwen Zero-shot performance evaluation (MMLU)

In [6]:
import pandas as pd

data = pd.read_csv("../data/mmlu/dev/abstract_algebra_dev.csv")
print(data.head())

  Find all c in Z_3 such that Z_3[x]/(x^2 + c) is a field.           0  \
0  Statement 1 | If aH is an element of a factor ...        True, True   
1  Statement 1 | Every element of a group generat...        True, True   
2  Statement 1| Every function from a finite set ...        True, True   
3            Find the characteristic of the ring 2Z.                 0   

              1            2            3  B  
0  False, False  True, False  False, True  B  
1  False, False  True, False  False, True  C  
2  False, False  True, False  False, True  A  
3             3           12           30  A  


In [ ]:
import os
import json
import pandas as pd
import string
import re
import time
import torch
import json
import random
from transformers import AutoModelForCausalLM, AutoTokenizer
import implementation

# Load MMLU examples
folder_path = os.path.expanduser("../data/mmlu/dev")
small_model_path= os.path.expanduser("../models/Qwen_Qwen2.5-0.5B")
medium_model_path = os.path.expanduser("../models/Qwen_Qwen2.5-3B-Instruct")

# Load the small model and tokenizer
small_model = AutoModelForCausalLM.from_pretrained(small_model_path, trust_remote_code=True)
small_tokenizer = AutoTokenizer.from_pretrained(small_model_path, trust_remote_code=True)

# Load the medium model and tokenizer
medium_model = AutoModelForCausalLM.from_pretrained(medium_model_path, trust_remote_code=True)
medium_tokenizer = AutoTokenizer.from_pretrained(medium_model_path, trust_remote_code=True)
medium_tokenizer.padding_side = "left"

mmlu_examples = []
for filename in os.listdir(folder_path):
    if filename.endswith(".csv"):
        data = pd.read_csv(os.path.join(folder_path, filename))
        # Format example as string prompts

        for _, row in data.iterrows():
            question = row.iloc[0]
            options = [row.iloc[1], row.iloc[2], row.iloc[3], row.iloc[4]]
            answer = row.iloc[5]

            string_prompt = f"Question: {question}\nOptions:\nA. {options[0]}\nB. {options[1]}\nC. {options[2]}\nD. {options[3]}\nAnswer:"        
            
            mmlu_examples.append((string_prompt, answer)) 
        

# String matching evaluation
tp_count = 0
fp_count = 0
failed_parse_count = 0

generation_times = []
all_results = []
incorrect_examples = []

for string_prompt, ground in mmlu_examples:

    if not string_prompt:
        continue

    # Tokenize
    prediction = medium_tokenizer(string_prompt, return_tensors="pt", padding=True)
    prediction_length = prediction.input_ids.shape[1]

    if torch.cuda.is_available():
        torch.cuda.synchronize()
    start_time = time.perf_counter()

    # Generate
    prediction_tensors = medium_model.generate(**prediction, max_new_tokens=256, pad_token_id=medium_tokenizer.eos_token_id)

    if torch.cuda.is_available():
        torch.cuda.synchronize()
    end_time = time.perf_counter()

    time_taken = end_time - start_time
    generation_times.append(time_taken)

    # Decode
    generated_tokens = prediction_tensors[0][prediction_length:]
    model_output = medium_tokenizer.decode(generated_tokens, skip_special_tokens=True)

    upper_model_output = model_output.upper()

    prediction = mmlu_baseline(None, upper_model_output)

    is_correct = prediction == ground

    if is_correct:
        tp_count += 1
    else:
        fp_count += 1
        if prediction is None:
            failed_parse_count += 1

        incorrect_examples.append({
            "prompt": string_prompt,
            "ground_truth": ground,
            "parsed_prediction": prediction,
            "raw_output": model_output
        })

    all_results.append({
        "prompt": string_prompt,
        "ground_truth": ground,
        "raw_model_generation": model_output,
        "parsed_prediction": prediction,
        "is_correct": is_correct
    })


# Aggregate results
total_questions = tp_count + fp_count

# Accuracy
accuracy_score = tp_count / total_questions if total_questions > 0 else 0.0

# Calcuate average generation time
average_time = sum(generation_times) / len(generation_times) if generation_times else 0

print(f"True Positives: {tp_count}")
print(f"False Positives: {fp_count}")
print(f"Failed to Parse: {failed_parse_count}")
print(f"Total Questions: {total_questions}")
print(f"Accuracy: {accuracy_score:.4f}")
print(f"Average Generation Time: {average_time:.2f} seconds per question")

output_filename = "Qwen_3B_results.json"

with open(output_filename, "w") as f:
    json.dump(all_results, f, indent=4)

num_to_sample = min(10, len(incorrect_examples))
if num_to_sample > 0:
    sampled_errors = random.sample(incorrect_examples, num_to_sample)
    for i, error in enumerate(sampled_errors):
        print(f"\n[Error {i+1}]")
        print(f"Ground Truth: {error['ground_truth']} | Parsed As: {error['parsed_prediction']}")
        print(f"Raw Output:\n{error['raw_output']}")
        print("-" * 40)


## GSM8k

In [ ]:
import os
import json
import pandas as pd
import string
import re
import time
import torch
import json
import random
from transformers import AutoModelForCausalLM, AutoTokenizer
from implementation import gsm8k_baseline

small_model_path= os.path.expanduser("../models/Qwen_Qwen2.5-0.5B")
medium_model_path = os.path.expanduser("../models/Qwen_Qwen2.5-3B-Instruct")

# Load the small model and tokenizer
small_model = AutoModelForCausalLM.from_pretrained(small_model_path, trust_remote_code=True)
small_tokenizer = AutoTokenizer.from_pretrained(small_model_path, trust_remote_code=True)

# Load the medium model and tokenizer
medium_model = AutoModelForCausalLM.from_pretrained(medium_model_path, trust_remote_code=True)
medium_tokenizer = AutoTokenizer.from_pretrained(medium_model_path, trust_remote_code=True)
medium_tokenizer.padding_side = "left"

# Load GSM8k examples
test_jsonl_path = os.path.expanduser("../data/gsm8k/test.jsonl")
train_jsonl_path = os.path.expanduser("../data/gsm8k/train.jsonl")

# Read jsonl files
gsm8k_examples = []
with open(train_jsonl_path, "r") as f:
    for line in f:
        train_data = json.loads(line)
        question = train_data["question"]
        raw_answer = train_data["answer"]

        ground_truth_number = raw_answer.split("####")[-1].strip()
        string_prompt = f"Question: {question}\nAnswer:"
        gsm8k_examples.append((string_prompt, ground_truth_number))


# String matching evaluation
tp_count = 0
fp_count = 0
failed_parse_count = 0
generation_times = []
all_results = []
incorrect_examples = []

for string_prompt, ground in gsm8k_examples:
    
    if not string_prompt:
        continue

    # Tokenize
    prediction = medium_tokenizer(string_prompt, return_tensors="pt", padding=True)
    prediction_length = prediction.input_ids.shape[1]

    if torch.cuda.is_available():
        torch.cuda.synchronize()
    start_time = time.perf_counter()

    # Generate
    prediction_tensors = medium_model.generate(**prediction, max_new_tokens=256, pad_token_id=medium_tokenizer.eos_token_id)

    if torch.cuda.is_available():
        torch.cuda.synchronize()
    end_time = time.perf_counter()

    time_taken = end_time - start_time
    generation_times.append(time_taken)

    # Decode
    generated_tokens = prediction_tensors[0][prediction_length:]
    model_output = medium_tokenizer.decode(generated_tokens, skip_special_tokens=True)

    upper_model_output = model_output.upper()

    prediction = gsm8k_baseline(None, upper_model_output)

    is_correct = prediction == ground

    if is_correct:
        tp_count += 1
    else:
        fp_count += 1
        if prediction is None:
            failed_parse_count += 1

        incorrect_examples.append({
            "prompt": string_prompt,
            "ground_truth": ground,
            "parsed_prediction": prediction,
            "raw_output": model_output
        })

    all_results.append({
        "prompt": string_prompt,
        "ground_truth": ground,
        "raw_model_generation": model_output,
        "parsed_prediction": prediction,
        "is_correct": is_correct
    })

# Aggregate results
total_questions = tp_count + fp_count

# Accuracy
accuracy_score = tp_count / total_questions if total_questions > 0 else 0.0

# Calcuate average generation time
average_time = sum(generation_times) / len(generation_times) if generation_times else 0

print(f"True Positives: {tp_count}")
print(f"False Positives: {fp_count}")
print(f"Failed to Parse: {failed_parse_count}")
print(f"Total Questions: {total_questions}")
print(f"Accuracy: {accuracy_score:.4f}")
print(f"Average Generation Time: {average_time:.2f} seconds per question")

output_filename = "Qwen_3B_gsm8k_results.json"
with open(output_filename, "w") as f:
    json.dump(all_results, f, indent=4)

num_to_sample = min(10, len(incorrect_examples))
if num_to_sample > 0:
    sampled_errors = random.sample(incorrect_examples, num_to_sample)
    for i, error in enumerate(sampled_errors):
        print(f"\n[Error {i+1}]")
        print(f"Ground Truth: {error['ground_truth']} | Parsed As: {error['parsed_prediction']}")
        print(f"Raw Output:\n{error['raw_output']}")
        print("-" * 40)

## Alpaca 


In [ ]:
import os
import json
import pandas as pd
import string
import re
import time
import torch
import json
import random
from transformers import AutoModelForCausalLM, AutoTokenizer

# More in the other file

## Alpaca Eval — Sampling Unsafe Baseline Outputs

In [2]:
import json
import random
import os

annotations_path = os.path.join(os.path.dirname(os.path.abspath(".")), 
                                "cs336_alignment/output/annotations.json")

with open(annotations_path) as f:
    annotations = json.load(f)

unsafe_examples = [ex for ex in annotations if ex["preference"] == 1.0]
print(f"Total annotations: {len(annotations)}")
print(f"Baseline loses (preference=1): {len(unsafe_examples)}")
print(f"Baseline wins  (preference=2): {len([e for e in annotations if e['preference'] == 2.0])}")
print(f"Annotator failed (preference=None): {len([e for e in annotations if e['preference'] is None])}")

random.seed(42)
sample = random.sample(unsafe_examples, 10)

for i, ex in enumerate(sample):
    print(f"\n{'='*70}")
    print(f"Example {i+1} | Dataset: {ex['dataset']}")
    print(f"{'='*70}")
    print(f"INSTRUCTION:\n  {ex['instruction'][:300]}")
    print(f"\nREFERENCE (GPT-4):\n  {ex['output_1'][:400]}")
    print(f"\nBASELINE (Qwen 0.5B):\n  {ex['output_2'][:400]}")


Total annotations: 805
Baseline loses (preference=1): 446
Baseline wins  (preference=2): 144
Annotator failed (preference=None): 215

Example 1 | Dataset: selfinstruct
INSTRUCTION:
  Provide a cooking hack for improving the flavor of the given food.

popcorn

REFERENCE (GPT-4):
  To improve the flavor of popcorn, try adding a combination of melted butter, garlic powder, and grated Parmesan cheese. Sprinkle this mixture over the popcorn before serving.

BASELINE (Qwen 0.5B):
  Sure, here's a simple recipe for making popcorn that will enhance its flavor:

Ingredients:
- 1 can of popcorn kernels
- 1/2 cup of butter
- 1/2 cup of water
- 1/4 cup of sugar
- 1/4 cup of salt

Instructions:
1. Preheat your oven to 350°F (175°C).
2. In a small bowl, mix together the butter, water, sugar, and salt.
3. Add the popcorn kernels to the bowl and stir until they are evenly coated with 

Example 2 | Dataset: helpful_base
INSTRUCTION:
  Did Facebook corporation change its name?

REFERENCE (GPT-4):
  No, 

## Look at Instruction Tuning Data

In [3]:
import os
from datasets import load_dataset

os.environ["HF_HOME"] = "/mnt/lustre/koa/scratch/kantas/hf_home"                                                                                                                                       
os.environ["HF_DATASETS_CACHE"] = "/mnt/lustre/koa/scratch/kantas/hf_home/datasets"                                                                                                                    
os.environ["TRANSFORMERS_CACHE"] = "/mnt/lustre/koa/scratch/kantas/hf_home/hub"  

train_dataset = load_dataset("HuggingFaceH4/ultrachat_200k", split="train_sft",                                                                                                                 
                               cache_dir="/mnt/lustre/koa/scratch/kantas/hf_home/datasets")                                                                                                              
test_dataset = load_dataset("HuggingFaceH4/ultrachat_200k", split="test_sft",                                                                                                                   
                              cache_dir="/mnt/lustre/koa/scratch/kantas/hf_home/datasets")

print(f"Train examples: {len(train_dataset)}")
print(f"Test examples: {len(test_dataset)}")

/home/kantas/koa_scratch/ece405-assignment3-alignment/.venv/lib/python3.12/site-packages/datasets/table.py:1421: FutureWarning: promote has been superseded by promote_options='default'.
  table = cls._concat_blocks(blocks, axis=0)


Train examples: 207865
Test examples: 23110


In [4]:
import random

random.seed(42)
indices = random.sample(range(len(train_dataset)), 10)

for i, idx in enumerate(indices):
    ex = train_dataset[idx]
    print(f"=== Example {i+1} (index {idx}) ===")
    for msg in ex["messages"]:
        role = msg["role"]
        content = msg["content"]
        if len(content) > 300:
            content = content[:300] + "..."
        print(f"  [{role}]: {content}")
    print()


=== Example 1 (index 167621) ===
  [user]: What are some traditional accompaniments to Laab Moo?
Generate according to: Laab Moo, Isan Thai Pork Salad (Click to enlarge) Laab Moo is a dish originating from the North-East (Isan) region of Thailand. It is popular throughout the country and you will find it at eateries from the very North to th...
  [assistant]: Some traditional accompaniments to Laab Moo include:

1. Sticky rice: Laab Moo is typically served with sticky rice, also known as glutinous rice. This type of rice is sticky and has a slightly sweet flavor that pairs well with the spicy and savory flavors of the pork salad.

2. Som Tum: Spicy Green...
  [user]: That sounds great! Can you please give me a step-by-step recipe for Spicy Green Papaya Salad (Som Tum)? I want to make it at home.
  [assistant]: Sure! Here's a step-by-step recipe for Spicy Green Papaya Salad (Som Tum):

Ingredients:
- 1 green papaya, peeled and shredded
- 2-3 cloves garlic, chopped
- 1-2 bird's eye chili